In [9]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/Screening_and_application


In [10]:
# dataname = "Arabidopsis_thaliana"
# dataname = 'Erigeron_breviscapus'
dataname = "Glycine_max"
# dataname = 'Zea_mays'

data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening2", f"{dataname}_data.pkl")
)
delete = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening2", f"{dataname}_deletedata.pkl")
)

In [11]:
alldata = pd.concat([data, delete], ignore_index=True)

In [12]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [13]:
bst = pd.read_pickle(
    os.path.join(
        CURRENT_DIR,
        "..",
        "data",
        "our_data",
        f"screening2/{dataname}deletedatamodel.dat",
    )
)

In [14]:
data_X, data_y = create_input_and_output_data(df=alldata)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
alldata["scores"] = y_test_pred
alldata.to_pickle(our_data + f"screening2/{dataname}_scores.pkl")

In [15]:
print(len(alldata))
print(alldata["enzyme"].nunique())
print(alldata["substrate"].nunique())

2403
342
7


In [20]:
TEST = alldata[(alldata["enzyme"] == "CYP72A69") & (alldata["substrate"] == "SOY")]

In [22]:
TEST["ESM1b"].values[0]

[-0.11592403054237366,
 0.1274290382862091,
 -0.03612707555294037,
 -0.005082077346742153,
 -0.16574768722057343,
 -0.2833961546421051,
 -0.3829319477081299,
 0.138337180018425,
 -0.1089886724948883,
 -0.24185645580291748,
 0.06955395638942719,
 -0.05941047891974449,
 0.1900366097688675,
 0.3319600522518158,
 0.2642316222190857,
 -0.008563892915844917,
 0.36396458745002747,
 0.03628333657979965,
 -0.0017999099800363183,
 -0.08117859810590744,
 0.054590076208114624,
 0.18410558998584747,
 -0.24627338349819183,
 0.21578414738178253,
 0.17364796996116638,
 -0.21241548657417297,
 -0.09097769856452942,
 -0.27171555161476135,
 -0.040932733565568924,
 -0.20197053253650665,
 -0.012957463972270489,
 -0.12709656357765198,
 0.17581354081630707,
 -0.03706414997577667,
 0.16214697062969208,
 -0.006631280295550823,
 0.022525686770677567,
 0.01519382931292057,
 -0.25392547249794006,
 0.3005431890487671,
 -0.03067212551832199,
 -0.04427564516663551,
 0.20038120448589325,
 -0.09048768132925034,
 -0.099